In [1]:
# Cell: Mount Google Drive (required if running this notebook fresh,
# separately from 01_Data_Engineering.ipynb)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Load Processed Data — Start of 02_Baseline_Model.ipynb
#
# Loads source_df, saved at the end of 01_Data_Engineering.ipynb.
# Rebuilds X_source/y_source using the exact same column-dropping
# logic used when they were first created (Cell 13), so behaviour
# is identical to running everything in one notebook.

import pandas as pd
import numpy as np

processed_dir = '/content/drive/MyDrive/ashrae_outputs/processed_data/'

print("Loading source_df from Drive...")
source_df = pd.read_parquet(processed_dir + 'source_df.parquet')
print(f"source_df loaded: {source_df.shape}")

# ── Rebuild X_source / y_source (identical logic to Cell 13)
cols_to_drop = [
    'meter_reading', 'meter',
    'building_id', 'site_id', 'primary_use',
    'timestamp',
    'square_feet'        # replaced by log_square_feet
]
cols_to_drop = [c for c in cols_to_drop if c in source_df.columns]

y_source = np.log1p(source_df['meter_reading'])
X_source = source_df.drop(columns=cols_to_drop)

print(f"Feature set ({len(X_source.columns)} features) ready.")

Loading source_df from Drive...
source_df loaded: (4412267, 26)
Feature set (19 features) ready.


In [3]:
# Cell 14: Base Model Hyperparameter Sensitivity Check
#
# Runs a small grid over the Base Model's key hyperparameters
# (max_depth, learning_rate), evaluated on the same Education
# time-based validation split used in Cell 15.
#
# Purpose: justify the final Base Model configuration with evidence
# rather than an unexplained choice. This does NOT replace Cell 15
# — it runs first, informs the choice, and Cell 15 still trains the
# final model everything else in the pipeline depends on.
#
# Note: uses fewer estimators (500) than the final model (2000) to
# keep the grid search fast — only relative ranking between
# configurations matters here, not final production performance.

import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

# ── Same time-based split as Cell 15
source_df_sorted = source_df.sort_values('timestamp')
cutoff_date = '2016-11-01'
train_mask  = source_df_sorted['timestamp'] < cutoff_date

X_tr = X_source.loc[source_df_sorted[train_mask].index]
X_v  = X_source.loc[source_df_sorted[~train_mask].index]
y_tr = y_source.loc[source_df_sorted[train_mask].index]
y_v  = y_source.loc[source_df_sorted[~train_mask].index]

# ── Grid definition
# Kept intentionally small (9 combinations) — this is a sensitivity
# check, not an exhaustive search. Centred on the values used in Cell 15.
max_depth_grid     = [4, 6, 8]
learning_rate_grid = [0.03, 0.05, 0.1]

grid_results = []

print("Running hyperparameter sensitivity grid (9 configurations)...")
print("Using n_estimators=500 for speed — final model uses 2000.\n")

for depth in max_depth_grid:
    for lr in learning_rate_grid:
        model = xgb.XGBRegressor(
            n_estimators       = 500,
            learning_rate      = lr,
            max_depth          = depth,
            subsample          = 0.8,
            colsample_bytree   = 0.8,
            reg_alpha          = 0.1,
            reg_lambda         = 1.0,
            eval_metric        = 'rmse',
            tree_method        = 'hist',
            device             = 'cuda',
            early_stopping_rounds = 30,
            random_state       = 42
        )
        model.fit(
            X_tr, y_tr,
            eval_set = [(X_v, y_v)],
            verbose  = False
        )

        preds_log = model.predict(X_v)
        y_true    = np.expm1(y_v.values)
        y_pred    = np.expm1(preds_log)

        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2   = r2_score(y_true, y_pred)

        grid_results.append({
            'max_depth'    : depth,
            'learning_rate': lr,
            'best_iteration': model.best_iteration,
            'RMSE (kWh)'   : round(rmse, 2),
            'R²'           : round(r2, 4)
        })

        print(f"  depth={depth}, lr={lr:.2f}  →  RMSE={rmse:.2f} kWh, R²={r2:.4f}, "
              f"stopped at tree {model.best_iteration}")

# ── Results table
grid_df = pd.DataFrame(grid_results).sort_values('RMSE (kWh)').reset_index(drop=True)

print("\n" + "="*70)
print("HYPERPARAMETER SENSITIVITY — RESULTS (sorted by RMSE, best first)")
print("="*70)
print(grid_df.to_string(index=False))
print("="*70)
print(f"\nBest configuration: max_depth={grid_df.iloc[0]['max_depth']}, "
      f"learning_rate={grid_df.iloc[0]['learning_rate']}")
print("Cell 15's final configuration (max_depth=6, learning_rate=0.05, "
      "n_estimators=2000) should be compared against this ranking.")

grid_df.to_csv('/content/hyperparameter_sensitivity.csv', index=False)
print("\nSaved to hyperparameter_sensitivity.csv")

Running hyperparameter sensitivity grid (9 configurations)...
Using n_estimators=500 for speed — final model uses 2000.



/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [23:55:35] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  depth=4, lr=0.03  →  RMSE=289.34 kWh, R²=0.6082, stopped at tree 499
  depth=4, lr=0.05  →  RMSE=284.93 kWh, R²=0.6200, stopped at tree 499
  depth=4, lr=0.10  →  RMSE=272.64 kWh, R²=0.6521, stopped at tree 499
  depth=6, lr=0.03  →  RMSE=268.03 kWh, R²=0.6638, stopped at tree 499
  depth=6, lr=0.05  →  RMSE=257.83 kWh, R²=0.6889, stopped at tree 499
  depth=6, lr=0.10  →  RMSE=248.44 kWh, R²=0.7111, stopped at tree 499
  depth=8, lr=0.03  →  RMSE=249.27 kWh, R²=0.7092, stopped at tree 499
  depth=8, lr=0.05  →  RMSE=245.39 kWh, R²=0.7182, stopped at tree 499
  depth=8, lr=0.10  →  RMSE=246.50 kWh, R²=0.7156, stopped at tree 499

HYPERPARAMETER SENSITIVITY — RESULTS (sorted by RMSE, best first)
 max_depth  learning_rate  best_iteration  RMSE (kWh)     R²
         8           0.05             499      245.39 0.7182
         8           0.10             499      246.50 0.7156
         6           0.10             499      248.44 0.7111
         8           0.03             499      249

In [4]:
# Cell 15: Base Model Training — saves teacher_model.json to Drive
#
# Trains the Base Model (teacher) on the Education source domain
# only. Saved directly to Drive, not /content/, since
# 03_Transfer_Learning.ipynb may run in a separate Colab session
# where /content/ is wiped.

import xgboost as xgb
import numpy as np
import os
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ── Full metric report
def evaluate(y_true_log, y_pred_log, label="Model"):
    """
    Accepts log-space arrays, converts to kWh, returns a metric dict.
    Reports RMSE, MAE, and R² — three complementary perspectives:
      RMSE  — penalises large errors heavily (sensitive to spikes)
      MAE   — average absolute error (robust, human-readable)
      R²    — proportion of variance explained (0–1 scale)
    """
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    rmse   = np.sqrt(mean_squared_error(y_true, y_pred))
    mae    = mean_absolute_error(y_true, y_pred)
    r2     = r2_score(y_true, y_pred)
    print(f"\n  [{label}]")
    print(f"    RMSE : {rmse:>10.2f} kWh")
    print(f"    MAE  : {mae:>10.2f} kWh")
    print(f"    R²   : {r2:>10.4f}")
    return {'rmse': rmse, 'mae': mae, 'r2': r2}

# ── Time-based split on Source (Education)
# Sort by timestamp; train on the first 10 months (Jan–Oct),
# validate on the final 2 months (Nov–Dec).
source_df_sorted = source_df.sort_values('timestamp')

cutoff_date = '2016-11-01'    # ASHRAE data covers the full year 2016
train_mask  = source_df_sorted['timestamp'] < cutoff_date

X_train = X_source.loc[source_df_sorted[train_mask].index]
X_val   = X_source.loc[source_df_sorted[~train_mask].index]
y_train = y_source.loc[source_df_sorted[train_mask].index]
y_val   = y_source.loc[source_df_sorted[~train_mask].index]

print(f"Time-based split:")
print(f"  Train : {len(X_train):>10,} rows  (Jan – Oct 2016)")
print(f"  Val   : {len(X_val):>10,} rows  (Nov – Dec 2016)")

# ── Base XGBoost Model
# A model that generalises well within Education will transfer
# better to the target domains.
print("\nInitialising Base XGBoost Model...")
base_model = xgb.XGBRegressor(
    n_estimators       = 2000,
    learning_rate      = 0.05,    # Slower = better generalisation before transfer
    max_depth          = 6,
    subsample          = 0.8,     # Train each tree on 80% of rows (random)
    colsample_bytree   = 0.8,     # Train each tree on 80% of features (random)
    reg_alpha          = 0.1,     # L1 regularisation — drives weak features to zero
    reg_lambda         = 1.0,     # L2 regularisation — shrinks all feature weights
    eval_metric        = 'rmse',
    tree_method        = 'hist',
    device             = 'cuda',  # Colab Pro GPU acceleration
    early_stopping_rounds = 50,
    random_state       = 42
)

print("Training Base Model on Education domain...")
base_model.fit(
    X_train, y_train,
    eval_set = [(X_train, y_train), (X_val, y_val)],
    verbose  = 100
)

print("\n--- Base Model Evaluation (Education Domain) ---")
val_preds_log = base_model.predict(X_val)
base_val_metrics = evaluate(y_val.values, val_preds_log, "Base Model — Education Val")

# ── Save model — to Drive, so it survives across separate notebook sessions
models_dir = '/content/drive/MyDrive/ashrae_outputs/models/'
os.makedirs(models_dir, exist_ok=True)
base_model.save_model(models_dir + 'teacher_model.json')
print(f"Base model saved to {models_dir}teacher_model.json")

Time-based split:
  Train :  3,650,649 rows  (Jan – Oct 2016)
  Val   :    761,618 rows  (Nov – Dec 2016)

Initialising Base XGBoost Model...
Training Base Model on Education domain...
[0]	validation_0-rmse:1.51118	validation_1-rmse:1.50293
[100]	validation_0-rmse:0.76332	validation_1-rmse:0.89574
[200]	validation_0-rmse:0.70157	validation_1-rmse:0.85454
[300]	validation_0-rmse:0.65714	validation_1-rmse:0.82793
[400]	validation_0-rmse:0.62667	validation_1-rmse:0.80995
[500]	validation_0-rmse:0.60358	validation_1-rmse:0.79631
[600]	validation_0-rmse:0.58733	validation_1-rmse:0.78812
[700]	validation_0-rmse:0.57336	validation_1-rmse:0.78020
[800]	validation_0-rmse:0.56102	validation_1-rmse:0.77284
[900]	validation_0-rmse:0.55090	validation_1-rmse:0.76789
[1000]	validation_0-rmse:0.54117	validation_1-rmse:0.76311
[1100]	validation_0-rmse:0.53361	validation_1-rmse:0.75975
[1200]	validation_0-rmse:0.52669	validation_1-rmse:0.75671
[1300]	validation_0-rmse:0.51969	validation_1-rmse:0.75400
[

In [5]:
# Verification — End of 02_Baseline_Model.ipynb
#
# Confirms the trained model was saved correctly and is reloadable,
# and copies the hyperparameter sensitivity results to Drive (Cell 14
# only saved this to /content/, which would be lost between sessions).

import os
import shutil
import xgboost as xgb

models_dir = '/content/drive/MyDrive/ashrae_outputs/models/'
model_path = models_dir + 'teacher_model.json'

# ── Confirm the model file exists and is a reasonable size
if os.path.exists(model_path):
    size_kb = os.path.getsize(model_path) / 1024
    print(f"✓ teacher_model.json found ({size_kb:.1f} KB)")
else:
    print("✗ teacher_model.json NOT FOUND — check Cell 15 ran successfully.")

# ── Reload it fresh to confirm it isn't corrupted
try:
    check_model = xgb.XGBRegressor()
    check_model.load_model(model_path)
    n_trees = len(check_model.get_booster().get_dump())
    print(f"✓ Model reloads correctly — {n_trees} trees confirmed.")
except Exception as e:
    print(f"✗ Model failed to reload: {e}")

# ── Copy hyperparameter sensitivity results to Drive too
src = '/content/hyperparameter_sensitivity.csv'
dst = models_dir + 'hyperparameter_sensitivity.csv'
if os.path.exists(src):
    shutil.copy(src, dst)
    print(f"✓ hyperparameter_sensitivity.csv copied to Drive.")
else:
    print("✗ hyperparameter_sensitivity.csv not found — check Cell 14 ran successfully.")

print("\nSummary:")
print(f"  Base Model RMSE : {base_val_metrics['rmse']:.2f} kWh")
print(f"  Base Model MAE  : {base_val_metrics['mae']:.2f} kWh")
print(f"  Base Model R²   : {base_val_metrics['r2']:.4f}")
print(f"  Stopping iteration : {base_model.best_iteration}")
print(f"  Total trees saved  : {n_trees}")
print("\n02_Baseline_Model.ipynb complete. Ready for 03_Transfer_Learning.ipynb.")

✓ teacher_model.json found (9830.1 KB)
✓ Model reloads correctly — 1376 trees confirmed.
✓ hyperparameter_sensitivity.csv copied to Drive.

Summary:
  Base Model RMSE : 245.88 kWh
  Base Model MAE  : 102.59 kWh
  Base Model R²   : 0.7171
  Stopping iteration : 1325
  Total trees saved  : 1376

02_Baseline_Model.ipynb complete. Ready for 03_Transfer_Learning.ipynb.
